In [ ]:
%load_ext autoreload
%autoreload 2

# Tabel 110 `minema` + kohakäändelised obliikvid

Lisaandmetena võib kasutada notebookiga 010 kokku kogutud statistikat.

Tasakaalus korpusest kogutakse kokku verbi `minema` esinemused, kus verbi vahetute alluvate seas on vähemalt üks kohakäändeline obliikvalluv. Iga kohakäändeline obliikvalluv antakse eraldi real koos võimaliku subjektiga.

In [ ]:
import set_root
import pandas as pd
from config import DATA_DERIVED_QUERY_RESULTS_DIR, METADATA_TSV_DELIMITER, METADATA_TSV_FILE, SOURCE_CONLLU_FILE, TMP_DIR
from data_helpers.meta import enrich_collection_with_metadata
from data_helpers.syntax.conllu_reader import CoNNLUReader
from data_helpers.syntax.render_graphviz import render_syntax_graph
from datetime import datetime

collection_output_file = DATA_DERIVED_QUERY_RESULTS_DIR / "110_notebook_collection.tsv"


In [ ]:
my_reader = CoNNLUReader(SOURCE_CONLLU_FILE)
my_verbs = ["minema"]

In [ ]:
%%time
from data_helpers.syntax.constants import SUBJECT_DEPRELS, VERB_POS, LOCATIVE_CASES
from data_helpers.syntax.list_utils import ListUtils
date_time = datetime.now().strftime("%Y%m%d-%H%M%S")

collected_data = []
count = 0
for collection_id, graph in my_reader.get_sentences():
    # matrix for node distances
    dpath = graph.get_distances_matrix()
    
    # verb nodes
    verb_nodes = [v for v in graph.get_nodes_by_attributes(attrname="POS", attrvalue=VERB_POS) if graph.nodes[v]["lemma"] in my_verbs]
    if not len(verb_nodes): 
        continue
    
    # obl nodes
    obl_nodes = graph.get_nodes_by_attributes(attrname="deprel", attrvalue="obl")
    if not len(obl_nodes): 
        continue
    
   
    subj_nodes = graph.get_nodes_by_attributes(attrname="deprel", attrvalue=SUBJECT_DEPRELS)
    
    
    for verb in verb_nodes:
        # childnodes
        kids = [k for k in dpath[verb] if dpath[verb][k] == 1]
        obl_kids = ListUtils.list_intersection(kids, obl_nodes)
        subj_kids = ListUtils.list_intersection(kids, subj_nodes)
        
        for obl in obl_kids:
            obl_case = graph.get_node_case(obl)
            if not obl_case in LOCATIVE_CASES:
                continue
            
            
            render_syntax_graph(graph=graph, highlight=[verb, obl], output_dir=TMP_DIR)
            d = {
                'sent_id':  graph.get_metadata('sent_id'),
                'text_uid':  graph.get_metadata('text_uid'),
                
                'verb':  graph.nodes[verb]["lemma"],
                'verb_feats':  " ".join(graph.nodes[verb]["feats"].keys()),
                
                'obl':  graph.nodes[obl]["lemma"],
                'obl_pos':  graph.get_node_pos(obl),
                'obl_case':  obl_case,
                
                'subj':  [graph.nodes[s]["form"] for s in subj_kids],
                'subj_pos':  [graph.get_node_pos(s) for s in subj_kids],
                'text': " ".join(
                        [graph.nodes[n]["form"] for n in sorted([verb] + [obl] + subj_kids)]
                ),
                'sentence_text':  graph.get_metadata('text'),
            }
            
            collected_data.append(d)

In [ ]:
collected_data

In [ ]:
df_collected = pd.DataFrame(collected_data)
if not df_collected.empty:
    for column in ["subj", "subj_pos"]:
        if column in df_collected.columns:
            df_collected[column] = df_collected[column].apply(lambda values: ", ".join(values) if isinstance(values, list) else values)

collection_output_file.parent.mkdir(parents=True, exist_ok=True)
df_collected.to_csv(collection_output_file, index=None, sep="\t")
df_collected

### Metaandmete lisamine

Metaandmete failist lisatakse koik veerud. Siin maaratakse ainult liitmise võtmeveerud.


In [ ]:
collected_uid_column = "text_uid"
metadata_uid_column = "uid"

metadata_output_file = collection_output_file.with_name(
    f"{collection_output_file.stem}_with_metadata.tsv"
)

df_enriched = enrich_collection_with_metadata(
    df_collected=df_collected,
    metadata_file=METADATA_TSV_FILE,
    collected_uid_column=collected_uid_column,
    metadata_uid_column=metadata_uid_column,
    output_file=metadata_output_file,
    metadata_delimiter=METADATA_TSV_DELIMITER,
    output_delimiter="\t",
)
df_enriched